<a href="https://colab.research.google.com/github/MaryamAmjad2/PyTorch/blob/main/Advance%20PyTorch/05_Modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torchvision import datasets,transforms

device="cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

# GET DATA

In [ ]:
# Get Data
import requests
import zipfile
from pathlib import Path

# Setup path to a data folder
data_path=Path('Data/')
image_path=data_path/"pizza_steak_sushi"

if image_path.is_dir():
  print("Exist")
else:
  print("Doesnt Exist")
  image_path.mkdir(parents=True,
                  exist_ok=True )

# Download Pizza Steak Sushi Data
with open (data_path/"pizza_steak_sushi.zip",'wb') as f:
  request=requests.get('https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip')
  print("Download Started")
  f.write(request.content)

# UnZip it
with zipfile.ZipFile(data_path/'pizza_steak_sushi.zip','r') as zip_ref:
  print("Unzipping File")
  zip_ref.extractall(image_path)
  print("DONE")


Exist
Download Started
Unzipping File
DONE


In [ ]:
train_dir=image_path/"train"
test_dir=image_path/'test'

#Create Dataset and Dataloader

In [ ]:
data_transform = transforms.Compose([

    # Resize images to 64X64
    transforms.Resize(size=(64,64)),

    # Flip the Image
    transforms.RandomHorizontalFlip(p=0.5),

    # Turn Image to Torch Tensor
    transforms.ToTensor()

])



In [ ]:
from torchvision import datasets
train_data=datasets.ImageFolder(root=train_dir,
                                transform=data_transform, #Transform data
                                target_transform=None) #Transform Targets/Label

test_data=datasets.ImageFolder(root=test_dir,
                               transform=data_transform
                               )


In [ ]:
class_dict=train_data.class_to_idx

#2.1 Create a DataSet and DataLoader( Python Script Mode)

using jupyter magic command .
* % Line Magic
* %% Write Content of cell to file


In [ ]:
# Create direcrtory for going modular scripts
import os
os.makedirs('going_modular', exist_ok=True)

In [ ]:
%%writefile going_modular/data_setup.py
'''
Contains Functionality for Crearing DataLoader for Image Classification Data'''
import os
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

NUM_WORKERS=os.cpu_count()
def create_dataloader(
    train_dir:str,
    test_dir:str,
    transform:transforms.Compose,
    batch_size:int,
    num_workers:int=NUM_WORKERS


):
  '''
  Create Train and test DataLoaders
  '''

  # 1. Create Datasets
  train_data=datasets.ImageFolder(root=train_dir,
                                  transform=transform)

  test_data=datasets.ImageFolder(root=test_dir,
                                transform=transform)


  #2. Get Class Names
  class_names=train_data.classes

  #3. DataLoaders
  train_dataloader=DataLoader(dataset=train_data,
                              batch_size=batch_size,
                              shuffle=True,
                              num_workers=NUM_WORKERS,
                              pin_memory=True
                              )

  test_dataloader=DataLoader(dataset=test_data,
                            batch_size=batch_size,
                            shuffle=False,
                            num_workers=NUM_WORKERS,
                            pin_memory=True
                            )
  return train_dataloader,test_dataloader,class_names




Overwriting going_modular/data_setup.py


pin_memory=True    speeds up data transfer from CPU to GPU by using a special kind of memory that GPUs can access faster.

In [ ]:
from going_modular  import data_setup
train_dataloader,test_data_loader,class_names=data_setup.create_dataloader(train_dir,test_dir,data_transform,32)

# 2.2 Turning Model Code in Python Script

In [ ]:
%%writefile going_modular/model_builder.py

'''Python Model Code to instantiate TinyVGG model
'''
import torch
from torch import nn

class TinyVGG(nn.Module):
  def __init__(self,input_shape:int,hidden_unit:int,output_shape:int)->None:
    super().__init__()

    #BLOCK 1

    self.conv_block_1=nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  padding=1,
                  stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  padding=1,
                  stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)

    )

    #BLOCK 2

    self.conv_block_2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  padding=1,
                  stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,
                  out_channels=hidden_unit,
                  kernel_size=3,
                  padding=1,
                  stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)

    )

    # Classifer Layer


    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_unit*16*16,
                  out_features=output_shape)
    )

  def forward(self,x):
    '''Better Way to Do is the One BElow CZ of Operator Fusion'''
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))






Overwriting going_modular/model_builder.py


In [ ]:
!python going_modular/model_builder.py

In [ ]:
import  torch
from going_modular import model_builder
device='cuda' if torch.cuda.is_available() else 'cpu'

# Instantiate a model
torch.manual_seed(42)
model_1=model_builder.TinyVGG(input_shape=3,hidden_unit=10,output_shape=len(class_names)).to(device)

model_1

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2560, out_features=3, bias=True)
  )
)

## Make prediction on Single Image

In [ ]:
img_batch,label_batch=next(iter(train_dataloader))
img_single,label_single=img_batch[0].unsqueeze(0),label_batch[0]

print(f'Single Image Shape {img_single.shape}')

# Perform Forward Pass
model_1.eval()
with torch.inference_mode():
  pred=model_1(img_single).to(device)
  prob=torch.softmax(pred,dim=1)
  label=torch.argmax(prob)

print(f'OutPut Logists :\n {pred}')
print(f'Output Prediction Probs : \n{prob}')
print(f'Output Label : {label}')
print(f'Actual Label : {label_single}')

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Single Image Shape torch.Size([1, 3, 64, 64])
OutPut Logists :
 tensor([[0.0578, 0.0634, 0.0351]])
Output Prediction Probs : 
tensor([[0.3352, 0.3371, 0.3277]])
Output Label : 1
Actual Label : 2


#3. Turn Train and Test into Python Script

In [ ]:
%%writefile going_modular/engine.py

'''
Turn Our Train_steps into Python Script It Should have acces to train_step and test_step
function
'''



import torch
from typing import Dict,List,Tuple
from tqdm.auto import tqdm

try:
    from torchmetrics import Accuracy
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "torchmetrics"])
    from torchmetrics import Accuracy


device='cuda' if torch.cuda.is_available() else 'cpu'

#1. Train Steps

def train_steps(model:torch.nn.Module,
                dataloader:torch.utils.data.DataLoader,
                loss_fn:torch.nn.Module,
                optimizer:torch.optim.Optimizer,
                accuracy_fn):

  model.train()
  train_loss,train_acc=0,0

  #Loop through dataloader
  for batch, (X,y) in enumerate(dataloader):

    # Send data to target device
    X,y=X.to(device),y.to(device)

    # Forward Pass
    y_pred=model(X) #Output Model Logits

    # Calculate Loss
    loss=loss_fn(y_pred,y)
    train_loss+=loss.item()

    optimizer.zero_grad()

    #Back Propagation
    loss.backward()

    optimizer.step()


    # Calculate Acccuracy
    y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
    train_acc += accuracy_fn(y_pred_class, y).item()

  # Adjust Metric to Get Avergae Loss
  train_loss=train_loss/len(dataloader)
  train_acc=train_acc/len(dataloader)
  return train_loss,train_acc



#2. Test steps

def  test_steps(model:torch.nn.Module,
                dataloader:torch.utils.data.DataLoader,
                loss_fn:torch.nn.Module,
                optimizer:torch.optim.Optimizer,
                accuracy_fn):

  # put in Eval Mode
  model.eval()

  test_loss,test_acc=0,0
  with torch.inference_mode():

    for batch,(X_test,y_test) in enumerate(dataloader):

      X_test,y_test=X_test.to(device),y_test.to(device)

      test_pred_logits=model(X_test)

      loss=loss_fn(test_pred_logits,y_test)
      test_loss+=loss.item()

      # Cal Accc
      test_pred_label=test_pred_logits.argmax(dim=1)
      test_acc += accuracy_fn(test_pred_label, y_test).item()
      #test_acc+=((test_pred_label==y_test).sum().item()/len(dataloader))
      #test_acc+=((test_pred_label==y_test).sum().item()/len(y_test))

      # Adjust Metric to Get Avergae Loss
  test_loss=test_loss/len(dataloader)
  test_acc=test_acc/len(dataloader)
  return test_loss, test_acc


# 3. Train Loop

def train_loop(model:torch.nn.Module,
                train_dataloader:torch.utils.data.DataLoader,
                test_dataloader:torch.utils.data.DataLoader,
                loss_fn:torch.nn.Module,
                optimizer:torch.optim.Optimizer,
               accurcay_fn,
               epochs:int = 6,
               device=device):

  # Empty Result Dictionary

  results={'train_loss':[],
           'train_acc':[],
           'test_loss':[],
           'test_acc':[]}


  for epoch in tqdm(range(epochs)):
    train_loss,train_acc=train_steps(model,train_dataloader,
                                     loss_fn,optimizer,accurcay_fn
                                    )

    test_loss,test_acc=test_steps(model,test_dataloader,
                                     loss_fn,optimizer,accurcay_fn
                                    )


    # print out

    print(f'Epoch: {epoch} | Train Loss {train_loss:.4f} | Train Accuracy {train_acc:.4f} | Test Loss {test_loss:.4f}  Test Accuracy {test_acc:.4f}')

    # Update Results Dict
    results['train_loss'].append(train_loss)
    results['train_acc'].append(train_acc)
    results['test_loss'].append(test_loss)
    results['test_acc'].append(test_acc)

  # Return the Filled Results
  return results















Overwriting going_modular/engine.py


# Save Model

In [ ]:
%%writefile going_modular/utils.py

# import torch
# from pathlib import Path

# def save_model(model:torch.nn.Module,
#                target_dir:str,
#                model_name:str):

#   # Create Target Directory
#   target_dir_path=Path(target_dir)
#   target_dir_path.mkdir(target_dir,exist_ok=True)

#   # Create Model Save Path
#   assert model_name.endswith('.pth') or model_name.endswith('.pt'),'name Should end on pth or pt'
#   model_save_path=target_dir_path/model_name

#   #Save the Model
#   print(f'[INFO] Saving Model to :{model_save_path}')
#   torch.save(obj=model.state_dict(),f=model_save_path)

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):

  # Create Target Directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  # Create Model Save Path
  assert model_name.endswith('.pth') or model_name.endswith('.pt'), 'Name should end with .pth or .pt'
  model_save_path = target_dir_path / model_name

  # Save the Model
  print(f'[INFO] Saving Model to: {model_save_path}')
  torch.save(obj=model.state_dict(), f=model_save_path)

Overwriting going_modular/utils.py


In [ ]:
!python going_modular/utils.py

In [ ]:
from going_modular import utils
utils.save_model(model_1,'Data','Test_model.pth')

[INFO] Saving Model to: Data/Test_model.pth


#4. Creating a Pyhton Script to Train ane  Eval Model

In [ ]:
%%writefile going_modular/train.py
'''
Train a model
'''

import os
import torch
from torch import nn
from torchvision import transforms
from timeit import default_timer as timer


import data_setup, engine,model_builder,utils

try:
    from torchmetrics import Accuracy
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "torchmetrics"])
    from torchmetrics import Accuracy


# Setup Hyperparameters
NUM_EPOCHS=5
BATCH_SIZE=32
HIDDEN_UNITS=10
LEARNING_RATE=0.001

#Setup Directory
train_dir='Data/pizza_steak_sushi/train'
test_dir='Data/pizza_steak_sushi/test'

#Device agnostic code
device='cuda' if torch.cuda.is_available() else 'cpu'

#Create Transform

data_transform=transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

#Create dataloaders and get Class names
train_dataloder,test_dataloader,class_names=data_setup.create_dataloader(train_dir,test_dir,
                             data_transform,BATCH_SIZE)


#Create Model

model_1=model_builder.TinyVGG(3,10,len(class_names)).to(device)

#Loss and Optimizer and Accuracy
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model_1.parameters(),lr=LEARNING_RATE)
accuracy_fn = Accuracy(task="multiclass", num_classes=len(class_names)).to(device)


start_time=timer()

#Start Training
results=engine.train_loop(model_1,train_dataloder,
                  test_dataloader,loss_fn,
                  optimizer,accuracy_fn,
                  6,device)

end_time=timer()

#Calculate Total Time
total_time=end_time-start_time
print(f'Total Time Taken: {total_time:.3f}')

#Save The Model to File

utils.save_model(model_1,'Models','Model_1.pth')


Overwriting going_modular/train.py


In [ ]:
!python going_modular/train.py


  0% 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch: 0 | Train Loss 1.0939 | Train Accuracy 0.4297 | Test Loss 1.1279  Test Accuracy 0.2604
 17% 1/6 [00:02<00:10,  2.02s/it]Epoch: 1 | Train Loss 1.0805 | Train Accuracy 0.4258 | Test Loss 1.1255  Test Accuracy 0.2604
 33% 2/6 [00:04<00:08,  2.01s/it]Epoch: 2 | Train Loss 1.1215 | Train Accuracy 0.3047 | Test Loss 1.1338  Test Accuracy 0.2604
 50% 3/6 [00:06<00:06,  2.02s/it]Epoch: 3 | Train Loss 1.1007 | Train Accuracy 0.3047 | Test Loss 1.0857  Test Accuracy 0.2604
 67% 4/6 [00:08<00:04,  2.05s/it]Epoch: 4 | Train Loss 1.0800 | Train Accuracy 0.4375 | Test Loss 1.0529  Test Accuracy 0.5417
 83% 5/6 [00:10<00:02,  2.27s/it]Epoch: 5 | Train Loss 1.1149 | Train Accuracy 0.2812 | Test Loss 1.0447  Test Accuracy 0.5417
100% 6/6 [00:13<00:00,